# SDG 3 Indicator Text Classification
**Group Assignment 2 — Complete Experiment Pipeline**

Run this notebook top-to-bottom in Google Colab. All dependencies install in Section 0.

In [ ]:
# Section 0: Install dependencies and configure paths
import subprocess, sys, os

# Install all requirements
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)

# NLTK downloads
import nltk
for pkg in ['punkt', 'stopwords', 'wordnet', 'punkt_tab', 'averaged_perceptron_tagger']:
    nltk.download(pkg, quiet=True)

# Make sure src/ and models/ are importable
if '.' not in sys.path:
    sys.path.insert(0, '.')

print('Setup complete.')


In [ ]:
# Download GloVe vectors (needed for Experiment 11)
# ~820MB — only downloads if not already present
import os, subprocess
if not os.path.exists('glove.6B.300d.txt'):
    print('Downloading GloVe 6B vectors (~820MB)...')
    subprocess.run(['wget', '-q', 'https://nlp.stanford.edu/data/glove.6B.zip'], check=True)
    subprocess.run(['unzip', '-q', 'glove.6B.zip', 'glove.6B.300d.txt'], check=True)
    os.remove('glove.6B.zip')
    print('GloVe ready.')
else:
    print('GloVe vectors already present.')


---
## Section 1 — Person A: EDA & Preprocessing
*Kayonga Elvis* — dataset understanding, text statistics, preprocessing pipeline.

Outputs: `data/devex_train_clean.csv`, `data/devex_test_clean.csv`, visualizations in `outputs/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

train_df = pd.read_csv('data/Devex_train.csv', encoding='latin-1', low_memory=False)
test_df  = pd.read_csv('data/Devex_test_questions.csv', encoding='latin-1', low_memory=False)

print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
train_df.head(3)


In [ ]:
# Column detection
def detect_text_col(df):
    obj_cols = [c for c in df.columns if df[c].dtype == object]
    return max(obj_cols, key=lambda c: df[c].dropna().astype(str).str.len().mean())

def detect_id_col(df):
    # prefer columns whose name suggests an ID
    for c in df.columns:
        if any(kw in c.lower() for kw in ('id', 'key', 'uid', 'uuid')):
            return c
    # fallback: first column with all-unique values that isn't object type
    for c in df.columns:
        if df[c].dtype != object and df[c].nunique() == len(df):
            return c
    return df.columns[0]

def detect_label_cols(df, text_col, id_col):
    import re
    return [c for c in df.columns
            if c not in (text_col, id_col)
            and df[c].dropna().astype(str).str.contains(r'\d+\.\d+', regex=True).mean() > 0.3]

TEXT_COL   = detect_text_col(train_df)
ID_COL     = detect_id_col(train_df)
LABEL_COLS = detect_label_cols(train_df, TEXT_COL, ID_COL)

print(f'Text column : {TEXT_COL}')
print(f'ID column   : {ID_COL}')
print(f'Label cols  : {len(LABEL_COLS)} -> {LABEL_COLS[:3]} ...')


In [ ]:
# Discover unique labels
from sklearn.preprocessing import MultiLabelBinarizer

def build_label_lists(df, label_cols):
    rows = []
    for _, row in df[label_cols].iterrows():
        labels = [str(v).strip() for v in row if pd.notna(v) and str(v).strip() not in ('', 'NA', 'nan')]
        rows.append(labels)
    return rows

label_lists = build_label_lists(train_df, LABEL_COLS)
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(label_lists)
ALL_LABELS = list(mlb.classes_)

print(f'Unique labels: {len(ALL_LABELS)}')
print(f'Label matrix : {Y.shape}')


In [ ]:
# Label distribution
import os
os.makedirs('outputs', exist_ok=True)

label_counts = Y.sum(axis=0)
sorted_idx = np.argsort(label_counts)[::-1]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(ALL_LABELS)), label_counts[sorted_idx], color='steelblue')
ax.set_xticks(range(len(ALL_LABELS)))
ax.set_xticklabels([ALL_LABELS[i][:20] for i in sorted_idx], rotation=90, fontsize=7)
ax.set_title('Label Frequency Distribution (SDG 3 Indicators)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('outputs/label_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Imbalance ratio: {label_counts.max() / label_counts.min():.2f}')


In [ ]:
# Text statistics
doc_lengths = train_df[TEXT_COL].dropna().astype(str).str.split().str.len()
print(f'Avg tokens : {doc_lengths.mean():.1f}')
print(f'Median     : {doc_lengths.median():.0f}')
print(f'Min / Max  : {doc_lengths.min()} / {doc_lengths.max()}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(doc_lengths, bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Document Length (tokens)')
axes[0].set_xlabel('Tokens')
char_lengths = train_df[TEXT_COL].dropna().astype(str).str.len()
axes[1].hist(char_lengths, bins=50, color='coral', edgecolor='white')
axes[1].set_title('Document Length (characters)')
axes[1].set_xlabel('Characters')
plt.tight_layout()
plt.savefig('outputs/document_length_histogram.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Preprocessing pipeline (Person A)
import re
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

DOMAIN_ACRONYMS = {'sdg', 'who', 'hiv', 'tb', 'usaid', 'ngo', 'oda', 'oecd', 'wash', 'ncds'}
DOMAIN_ACRONYMS_UPPER = {a.upper() for a in DOMAIN_ACRONYMS}
STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
tokenizer  = RegexpTokenizer(r'\b[a-zA-Z]+\b')

def clean_html(text):
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'&[a-z]+;', ' ', text)
    return text

def preprocess_text(text):
    if pd.isna(text): return ''
    text = clean_html(str(text))
    text = text.lower()
    tokens = tokenizer.tokenize(text)
    cleaned = []
    for tok in tokens:
        if tok.upper() in DOMAIN_ACRONYMS_UPPER:
            cleaned.append(tok.upper())
            continue
        if tok in STOP_WORDS: continue
        lemma = lemmatizer.lemmatize(tok)
        if len(lemma) > 1:
            cleaned.append(lemma)
    return ' '.join(cleaned)

# Skip if clean CSVs already exist
if not os.path.exists('data/devex_train_clean.csv'):
    print('Preprocessing train...')
    train_df['clean_text'] = train_df[TEXT_COL].apply(preprocess_text)
    train_df.to_csv('data/devex_train_clean.csv', index=False)
    print('Preprocessing test...')
    test_df['clean_text'] = test_df[TEXT_COL].apply(preprocess_text)
    test_df.to_csv('data/devex_test_clean.csv', index=False)
    print('Clean CSVs saved.')
else:
    print('Clean CSVs already exist — skipping preprocessing.')


---
## Section 2 — Person B: Feature Engineering & Classical ML (Experiments 1–6)
*Nformi Modestine* — TF-IDF/BoW feature engineering, classical ML model comparison, threshold optimisation, class imbalance handling.

**Best result this section: Experiment 4 — LinearSVC, Hamming Loss = 0.045**

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import hamming_loss, f1_score, accuracy_score
from sklearn.preprocessing import MultiLabelBinarizer

TRAIN_PATH = 'data/devex_train_clean.csv'
TEST_PATH  = 'data/devex_test_clean.csv'

train_df = pd.read_csv(TRAIN_PATH, encoding='latin-1', low_memory=False)
test_df  = pd.read_csv(TEST_PATH,  encoding='latin-1', low_memory=False)
print(f'Train: {train_df.shape}, Test: {test_df.shape}')

In [ ]:
\
# Reconstruct label matrix from clean CSV
def detect_label_columns(df, text_col='clean_text'):
    return [c for c in df.columns
            if c not in (text_col, 'clean_text')
            and df[c].dropna().astype(str).str.contains(r'\d+\.\d+', regex=True).mean() > 0.3]

RAW_TEXT_COL = [c for c in train_df.columns if 'text' in c.lower() and c != 'clean_text'][0]
LABEL_COLS_B = detect_label_columns(train_df)

def build_label_lists(df, label_cols):
    rows = []
    for _, row in df[label_cols].iterrows():
        labels = [str(v).strip() for v in row if pd.notna(v) and str(v).strip() not in ('', 'NA', 'nan')]
        rows.append(labels)
    return rows

mlb_b = MultiLabelBinarizer()
label_lists_b = build_label_lists(train_df, LABEL_COLS_B)
Y_b = mlb_b.fit_transform(label_lists_b)
ALL_LABELS_B = list(mlb_b.classes_)

train_df['clean_text'] = train_df['clean_text'].fillna('')
X_b = train_df['clean_text'].values

X_train_b, X_val_b, y_train_b, y_val_b = train_test_split(
    X_b, Y_b, test_size=0.2, random_state=42)

print(f'Labels: {len(ALL_LABELS_B)}, Train: {len(X_train_b)}, Val: {len(X_val_b)}')\


In [ ]:
# Shared evaluation helper for Section 2
from src.evaluation import MultiLabelEvaluator
import os
os.makedirs('results/confusion_matrices', exist_ok=True)

experiment_results_b = {}

def evaluate_b(name, model, X_val, y_val, exp_num, y_prob=None):
    y_pred = model.predict(X_val)
    if y_prob is None:
        if hasattr(model, 'predict_proba'):
            y_prob = model.predict_proba(X_val)
        else:
            y_prob = y_pred.astype(float)
    hl     = hamming_loss(y_val, y_pred)
    f1_mi  = f1_score(y_val, y_pred, average='micro', zero_division=0)
    f1_ma  = f1_score(y_val, y_pred, average='macro', zero_division=0)
    print(f'  Exp {exp_num} {name}: HL={hl:.4f}  MicroF1={f1_mi:.4f}  MacroF1={f1_ma:.4f}')
    evaluator_b = MultiLabelEvaluator(label_names=ALL_LABELS_B)
    evaluator_b.save_confusion_matrices(y_val, y_prob.astype(float), threshold=0.5, exp_num=exp_num)
    experiment_results_b[exp_num] = {
        'name': name, 'hl': hl, 'f1_micro': f1_mi, 'f1_macro': f1_ma,
        'y_pred': y_pred, 'y_prob': y_prob
    }
    return hl, f1_mi, f1_ma

### Experiment 1 — LR + TF-IDF Unigrams (Baseline)

In [ ]:
print('Running Experiment 1: LR + TF-IDF Unigrams (Baseline)')
vec1 = TfidfVectorizer(ngram_range=(1,1), max_features=10000)
X_tr1 = vec1.fit_transform(X_train_b)
X_v1  = vec1.transform(X_val_b)
lr1 = OneVsRestClassifier(LogisticRegression(max_iter=1000, C=1.0))
lr1.fit(X_tr1, y_train_b)
evaluate_b('LR + TF-IDF Unigrams', lr1, X_v1, y_val_b, exp_num=1)

### Experiment 2 — LR + TF-IDF Bigrams

In [ ]:
print('Running Experiment 2: LR + TF-IDF Bigrams')
vec2 = TfidfVectorizer(ngram_range=(1,2), max_features=10000)
X_tr2 = vec2.fit_transform(X_train_b)
X_v2  = vec2.transform(X_val_b)
lr2 = OneVsRestClassifier(LogisticRegression(max_iter=1000, C=1.0))
lr2.fit(X_tr2, y_train_b)
evaluate_b('LR + TF-IDF Bigrams', lr2, X_v2, y_val_b, exp_num=2)

### Experiment 3 — Vocabulary Size Tuning

In [ ]:
print('Running Experiment 3: Vocabulary Size Tuning')
for max_feat in [5000, 10000, 20000, 50000]:
    v = TfidfVectorizer(ngram_range=(1,1), max_features=max_feat)
    Xtr = v.fit_transform(X_train_b)
    Xv  = v.transform(X_val_b)
    m = OneVsRestClassifier(LogisticRegression(max_iter=1000))
    m.fit(Xtr, y_train_b)
    hl = hamming_loss(y_val_b, m.predict(Xv))
    print(f'  vocab={max_feat:6d} -> HL={hl:.4f}')
# Best was 5000 — use that going forward
vec3 = TfidfVectorizer(ngram_range=(1,1), max_features=5000)
X_tr3 = vec3.fit_transform(X_train_b)
X_v3  = vec3.transform(X_val_b)
lr3 = OneVsRestClassifier(LogisticRegression(max_iter=1000))
lr3.fit(X_tr3, y_train_b)
evaluate_b('LR + TF-IDF vocab=5000 (best)', lr3, X_v3, y_val_b, exp_num=3)

### Experiment 4 — Model Comparison: LR vs LinearSVC vs RF

In [ ]:
print('Running Experiment 4: Model Comparison')
models_exp4 = {
    'LR (best config)': OneVsRestClassifier(LogisticRegression(max_iter=1000)),
    'LinearSVC':        OneVsRestClassifier(LinearSVC(max_iter=2000)),
    'RandomForest':     OneVsRestClassifier(RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
}
results4 = {}
for name, m in models_exp4.items():
    m.fit(X_tr3, y_train_b)
    y_pred_tmp = m.predict(X_v3)
    hl  = hamming_loss(y_val_b, y_pred_tmp)
    f1  = f1_score(y_val_b, y_pred_tmp, average='micro', zero_division=0)
    results4[name] = (hl, f1, m)
    print(f'  {name:20s} HL={hl:.4f}  MicroF1={f1:.4f}')

best_name4  = min(results4, key=lambda k: results4[k][0])
best_model4 = results4[best_name4][2]
print(f'\nBest model: {best_name4}')
evaluate_b(best_name4, best_model4, X_v3, y_val_b, exp_num=4)

### Experiment 5 — Threshold Optimisation

In [ ]:
print('Running Experiment 5: Threshold Optimisation')
lr5 = OneVsRestClassifier(LogisticRegression(max_iter=1000))
lr5.fit(X_tr3, y_train_b)
y_prob5 = lr5.predict_proba(X_v3)

best_hl5, best_t5 = 1.0, 0.5
for t in np.arange(0.1, 0.9, 0.05):
    y_bin = (y_prob5 >= t).astype(int)
    hl = hamming_loss(y_val_b, y_bin)
    if hl < best_hl5:
        best_hl5, best_t5 = hl, t

print(f'  Best threshold={best_t5:.2f}  HL={best_hl5:.4f}')
y_pred5 = (y_prob5 >= best_t5).astype(int)
f1_5 = f1_score(y_val_b, y_pred5, average='micro', zero_division=0)
f1_ma5 = f1_score(y_val_b, y_pred5, average='macro', zero_division=0)
evaluator_b5 = MultiLabelEvaluator(label_names=ALL_LABELS_B)
evaluator_b5.save_confusion_matrices(y_val_b, y_prob5, threshold=best_t5, exp_num=5)
experiment_results_b[5] = {
    'name': 'LR + Opt Threshold', 'hl': best_hl5,
    'f1_micro': f1_5, 'f1_macro': f1_ma5,
    'y_pred': y_pred5, 'y_prob': y_prob5
}

### Experiment 6 — Class Imbalance: class_weight='balanced'

In [ ]:
print('Running Experiment 6: Class Imbalance Handling')
lr6 = OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight='balanced'))
lr6.fit(X_tr3, y_train_b)
evaluate_b('LR balanced', lr6, X_v3, y_val_b, exp_num=6)

---
## Section 3 — Person 3: Deep Learning with Word2Vec & BERT (Experiments 7–10)
*Patricia Mugabo* — Word2Vec embeddings, deep neural networks, BERT feature extraction, hyperparameter tuning, class weighting.

**Best result this section: Experiment 9 — Word2Vec + DeepNN (Tuned), Hamming Loss = 0.0595**

In [ ]:
import sys, os
sys.path.insert(0, '.')
import numpy as np
import torch
from src.data_loader import SDGDataLoader
from src.evaluation import MultiLabelEvaluator
from src.trainer import Trainer, TextDataset
from models.neural_networks import DeepNN
from models.embedding_models import Word2VecEmbedding

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

loader = SDGDataLoader(
    train_path='data/devex_train_clean.csv',
    test_path='data/devex_test_clean.csv',
    use_cleaned=True
)
train_df_dl, test_df_dl = loader.load_data()
train_df_dl, test_df_dl = loader.preprocess_dataset(text_column='text')
label_columns = loader.get_label_columns()

X_train_dl, X_val_dl, y_train_dl, y_val_dl = loader.train_val_split(test_size=0.2, random_state=42)
y_train_dl = y_train_dl.astype(float)
y_val_dl   = y_val_dl.astype(float)
print(f'Train: {len(X_train_dl)}  Val: {len(X_val_dl)}  Labels: {len(label_columns)}')

### Experiment 7 — Word2Vec (trained on corpus) + DeepNN

In [ ]:
print('Running Experiment 7: Word2Vec + DeepNN')

X_train_tok7 = [t.split() for t in X_train_dl]
X_val_tok7   = [t.split() for t in X_val_dl]

w2v7 = Word2VecEmbedding(vector_size=300, window=5, min_count=2, workers=4, sg=1)
w2v7.train(X_train_tok7)

X_tr7 = w2v7.transform(X_train_tok7)
X_v7  = w2v7.transform(X_val_tok7)

model7 = DeepNN(input_dim=300, hidden_dims=[512, 256, 128], output_dim=y_train_dl.shape[1],
                dropout=0.3, use_batch_norm=True)

evaluator7 = MultiLabelEvaluator(label_names=label_columns)
tr7 = Trainer(model7,
              torch.utils.data.DataLoader(TextDataset(X_tr7, y_train_dl), batch_size=64, shuffle=True),
              torch.utils.data.DataLoader(TextDataset(X_v7, y_val_dl),   batch_size=64),
              device=device)

os.makedirs('results/experiment_7_new', exist_ok=True)
model7 = tr7.train(num_epochs=30, evaluator=evaluator7, save_path='results/experiment_7_new')

model7.eval()
with torch.no_grad():
    preds7 = []
    for xb, _ in torch.utils.data.DataLoader(TextDataset(X_v7, y_val_dl), batch_size=64):
        preds7.append(model7(xb.to(device)).cpu().numpy())
y_pred7 = np.vstack(preds7)

from sklearn.metrics import hamming_loss, f1_score
hl7  = hamming_loss(y_val_dl, (y_pred7 >= 0.5).astype(int))
f1_7 = f1_score(y_val_dl, (y_pred7 >= 0.5).astype(int), average='micro', zero_division=0)
print(f'Exp 7 — HL={hl7:.4f}  MicroF1={f1_7:.4f}')

evaluator7.save_confusion_matrices(y_val_dl, y_pred7, threshold=0.5, exp_num=7)

### Experiment 8 — BERT (pretrained contextual embeddings)

In [ ]:
print('Running Experiment 8: BERT')
from transformers import BertTokenizer, BertModel

tokenizer_bert = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)
bert_model.eval()

def bert_encode(texts, batch_size=32):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = list(texts[i:i+batch_size])
        enc = tokenizer_bert(batch, padding=True, truncation=True,
                             max_length=128, return_tensors='pt').to(device)
        with torch.no_grad():
            out = bert_model(**enc)
        vecs = out.last_hidden_state[:, 0, :].cpu().numpy()
        all_vecs.append(vecs)
    return np.vstack(all_vecs)

print('Encoding train...')
X_tr8 = bert_encode(X_train_dl)
print('Encoding val...')
X_v8  = bert_encode(X_val_dl)
print(f'BERT embedding shape: {X_tr8.shape}')

model8 = DeepNN(input_dim=768, hidden_dims=[256, 128], output_dim=y_train_dl.shape[1],
                dropout=0.3, use_batch_norm=True)

evaluator8 = MultiLabelEvaluator(label_names=label_columns)
tr8 = Trainer(model8,
              torch.utils.data.DataLoader(TextDataset(X_tr8, y_train_dl), batch_size=32, shuffle=True),
              torch.utils.data.DataLoader(TextDataset(X_v8,  y_val_dl),   batch_size=32),
              device=device)

os.makedirs('results/experiment_8_new', exist_ok=True)
model8 = tr8.train(num_epochs=15, evaluator=evaluator8, save_path='results/experiment_8_new')

model8.eval()
with torch.no_grad():
    preds8 = []
    for xb, _ in torch.utils.data.DataLoader(TextDataset(X_v8, y_val_dl), batch_size=32):
        preds8.append(model8(xb.to(device)).cpu().numpy())
y_pred8 = np.vstack(preds8)

hl8  = hamming_loss(y_val_dl, (y_pred8 >= 0.5).astype(int))
f1_8 = f1_score(y_val_dl, (y_pred8 >= 0.5).astype(int), average='micro', zero_division=0)
print(f'Exp 8 — HL={hl8:.4f}  MicroF1={f1_8:.4f}')
evaluator8.save_confusion_matrices(y_val_dl, y_pred8, threshold=0.5, exp_num=8)

### Experiment 9 — Word2Vec + DeepNN (Tuned hyperparameters)

In [ ]:
print('Running Experiment 9: Word2Vec + DeepNN (Tuned)')
w2v9 = Word2VecEmbedding(vector_size=200, window=5, min_count=2, workers=4, sg=1)
w2v9.train(X_train_tok7)

X_tr9 = w2v9.transform(X_train_tok7)
X_v9  = w2v9.transform(X_val_tok7)

model9 = DeepNN(input_dim=200, hidden_dims=[512, 256, 128, 64], output_dim=y_train_dl.shape[1],
                dropout=0.2, use_batch_norm=True)

evaluator9 = MultiLabelEvaluator(label_names=label_columns)
tr9 = Trainer(model9,
              torch.utils.data.DataLoader(TextDataset(X_tr9, y_train_dl), batch_size=64, shuffle=True),
              torch.utils.data.DataLoader(TextDataset(X_v9,  y_val_dl),   batch_size=64),
              device=device, learning_rate=0.001)

os.makedirs('results/experiment_9_new', exist_ok=True)
model9 = tr9.train(num_epochs=30, evaluator=evaluator9, save_path='results/experiment_9_new')

model9.eval()
with torch.no_grad():
    preds9 = []
    for xb, _ in torch.utils.data.DataLoader(TextDataset(X_v9, y_val_dl), batch_size=64):
        preds9.append(model9(xb.to(device)).cpu().numpy())
y_pred9 = np.vstack(preds9)

hl9  = hamming_loss(y_val_dl, (y_pred9 >= 0.5).astype(int))
f1_9 = f1_score(y_val_dl, (y_pred9 >= 0.5).astype(int), average='micro', zero_division=0)
print(f'Exp 9 — HL={hl9:.4f}  MicroF1={f1_9:.4f}')
evaluator9.save_confusion_matrices(y_val_dl, y_pred9, threshold=0.5, exp_num=9)

### Experiment 10 — Word2Vec + Class Weights

In [ ]:
print('Running Experiment 10: Word2Vec + Class Weights')
label_counts_dl = y_train_dl.sum(axis=0)
n_samples = len(y_train_dl)
class_weights10 = (n_samples - label_counts_dl) / (label_counts_dl + 1e-8)
class_weights10 = np.clip(class_weights10, 1.0, 10.0)

model10 = DeepNN(input_dim=200, hidden_dims=[512, 256, 128, 64], output_dim=y_train_dl.shape[1],
                 dropout=0.2, use_batch_norm=True)

evaluator10 = MultiLabelEvaluator(label_names=label_columns)
tr10 = Trainer(model10,
               torch.utils.data.DataLoader(TextDataset(X_tr9, y_train_dl), batch_size=64, shuffle=True),
               torch.utils.data.DataLoader(TextDataset(X_v9,  y_val_dl),   batch_size=64),
               device=device, class_weights=class_weights10)

os.makedirs('results/experiment_10_new', exist_ok=True)
model10 = tr10.train(num_epochs=30, evaluator=evaluator10, save_path='results/experiment_10_new')

model10.eval()
with torch.no_grad():
    preds10 = []
    for xb, _ in torch.utils.data.DataLoader(TextDataset(X_v9, y_val_dl), batch_size=64):
        preds10.append(model10(xb.to(device)).cpu().numpy())
y_pred10 = np.vstack(preds10)

best_hl10, best_t10 = 1.0, 0.5
for t in np.arange(0.1, 0.9, 0.05):
    hl = hamming_loss(y_val_dl, (y_pred10 >= t).astype(int))
    if hl < best_hl10:
        best_hl10, best_t10 = hl, t

print(f'Exp 10 — Best HL={best_hl10:.4f} at threshold={best_t10:.2f}')
evaluator10.save_confusion_matrices(y_val_dl, y_pred10, threshold=best_t10, exp_num=10)

---
## Section 4 — Lorita: New Embedding Experiments (11–12)
Comparing pretrained static embeddings (GloVe) and subword-aware embeddings (FastText) against trained Word2Vec, using the same DeepNN architecture as Experiment 9 for fair comparison.

### Experiment 11 — GloVe (pretrained, 6B.300d) + DeepNN
**Hypothesis:** Pretrained embeddings from 6B tokens should give richer semantic representations than Word2Vec trained on ~3,000 documents.

In [ ]:
print('Running Experiment 11: GloVe + DeepNN')
from models.embedding_models import GloVeEmbedding
import os

GLOVE_PATH = 'glove.6B.300d.txt'
assert os.path.exists(GLOVE_PATH), f'GloVe file not found at {GLOVE_PATH}. Run the setup cell in Section 0.'

glove11 = GloVeEmbedding(embedding_dim=300)
glove11.load(GLOVE_PATH)

X_train_tok11 = [t.split() for t in X_train_dl]
X_val_tok11   = [t.split() for t in X_val_dl]

X_tr11 = glove11.transform(X_train_tok11)
X_v11  = glove11.transform(X_val_tok11)
print(f'GloVe embedding shape: {X_tr11.shape}')


In [ ]:
# Same architecture as Exp 9 (tuned) for fair comparison
model11 = DeepNN(input_dim=300, hidden_dims=[512, 256, 128, 64], output_dim=y_train_dl.shape[1],
                 dropout=0.2, use_batch_norm=True)

evaluator11 = MultiLabelEvaluator(label_names=label_columns)
tr11 = Trainer(model11,
               torch.utils.data.DataLoader(TextDataset(X_tr11, y_train_dl), batch_size=64, shuffle=True),
               torch.utils.data.DataLoader(TextDataset(X_v11,  y_val_dl),   batch_size=64),
               device=device, learning_rate=0.001)

os.makedirs('results/experiment_11', exist_ok=True)
model11 = tr11.train(num_epochs=30, evaluator=evaluator11, save_path='results/experiment_11')

model11.eval()
with torch.no_grad():
    preds11 = []
    for xb, _ in torch.utils.data.DataLoader(TextDataset(X_v11, y_val_dl), batch_size=64):
        preds11.append(model11(xb.to(device)).cpu().numpy())
y_pred11 = np.vstack(preds11)

from sklearn.metrics import hamming_loss, f1_score
hl11     = hamming_loss(y_val_dl, (y_pred11 >= 0.5).astype(int))
f1_11    = f1_score(y_val_dl, (y_pred11 >= 0.5).astype(int), average='micro', zero_division=0)
f1_mac11 = f1_score(y_val_dl, (y_pred11 >= 0.5).astype(int), average='macro', zero_division=0)
print(f'Exp 11 GloVe — HL={hl11:.4f}  MicroF1={f1_11:.4f}  MacroF1={f1_mac11:.4f}')

import json
with open('results/experiment_11/results.json', 'w') as f:
    json.dump({'experiment_id': 11, 'experiment_name': 'GloVe + DeepNN',
               'results': {'hamming_loss': hl11, 'f1_micro': f1_11, 'f1_macro': f1_mac11}}, f, indent=2)

evaluator11.save_confusion_matrices(y_val_dl, y_pred11, threshold=0.5, exp_num=11)
print('Experiment 11 complete.')


### Experiment 12 — FastText (trained on corpus, subword) + DeepNN
**Hypothesis:** Subword information helps represent rare/technical health terms (e.g. 'hepatitis', 'neonatal') that Word2Vec might miss due to low frequency.

In [ ]:
print('Running Experiment 12: FastText + DeepNN')
from models.embedding_models import FastTextEmbedding

ft12 = FastTextEmbedding(vector_size=300, window=5, min_count=1, workers=4)
ft12.train(X_train_tok11)

X_tr12 = ft12.transform(X_train_tok11)
X_v12  = ft12.transform(X_val_tok11)
print(f'FastText embedding shape: {X_tr12.shape}')

In [ ]:
# Same architecture as Exp 9/11 for fair comparison
model12 = DeepNN(input_dim=300, hidden_dims=[512, 256, 128, 64], output_dim=y_train_dl.shape[1],
                 dropout=0.2, use_batch_norm=True)

evaluator12 = MultiLabelEvaluator(label_names=label_columns)
tr12 = Trainer(model12,
               torch.utils.data.DataLoader(TextDataset(X_tr12, y_train_dl), batch_size=64, shuffle=True),
               torch.utils.data.DataLoader(TextDataset(X_v12,  y_val_dl),   batch_size=64),
               device=device, learning_rate=0.001)

os.makedirs('results/experiment_12', exist_ok=True)
model12 = tr12.train(num_epochs=30, evaluator=evaluator12, save_path='results/experiment_12')

model12.eval()
with torch.no_grad():
    preds12 = []
    for xb, _ in torch.utils.data.DataLoader(TextDataset(X_v12, y_val_dl), batch_size=64):
        preds12.append(model12(xb.to(device)).cpu().numpy())
y_pred12 = np.vstack(preds12)

from sklearn.metrics import hamming_loss, f1_score
hl12     = hamming_loss(y_val_dl, (y_pred12 >= 0.5).astype(int))
f1_12    = f1_score(y_val_dl, (y_pred12 >= 0.5).astype(int), average='micro', zero_division=0)
f1_mac12 = f1_score(y_val_dl, (y_pred12 >= 0.5).astype(int), average='macro', zero_division=0)
print(f'Exp 12 FastText — HL={hl12:.4f}  MicroF1={f1_12:.4f}  MacroF1={f1_mac12:.4f}')

import json
with open('results/experiment_12/results.json', 'w') as f:
    json.dump({'experiment_id': 12, 'experiment_name': 'FastText + DeepNN',
               'results': {'hamming_loss': hl12, 'f1_micro': f1_12, 'f1_macro': f1_mac12}}, f, indent=2)

evaluator12.save_confusion_matrices(y_val_dl, y_pred12, threshold=0.5, exp_num=12)
print('Experiment 12 complete.')

---
*End of Section 4. All 12 experiments defined. Proceed to Section 5 for the unified comparison table.*